In [1]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — Setup
# ═══════════════════════════════════════════════════════════════════
import subprocess, sys, os, time, warnings, re as _re
warnings.filterwarnings('ignore')
 
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'casanovo>=5.0.0', 'pyteomics', 'lxml', 'remotezip', 'appdirs'], check=True)
 
import numpy as np, pandas as pd, datetime
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from torch.profiler import profile, ProfilerActivity, schedule, record_function
from pathlib import Path
 
torch.manual_seed(42)
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_NAME   = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'
TOTAL_VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == 'cuda' else 0
N_PEAKS    = 150   # config default max_peaks
AVG_PEP    = 12    # median observed peptide length from dataset
 
print(f'Device: {DEVICE} | GPU: {GPU_NAME} | VRAM: {TOTAL_VRAM:.1f} GB')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')
os.makedirs('results', exist_ok=True)


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


Device: cuda | GPU: NVIDIA L4 | VRAM: 23.6 GB
PyTorch: 2.7.1+cu128 | CUDA: 12.8


In [2]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — Download MGF via HTTP range request (no full zip)
# ═══════════════════════════════════════════════════════════════════
from remotezip import RemoteZip
import shutil
 
ZIP_URL  = 'https://zenodo.org/records/12587317/files/mgf_data.zip?download=1'
TARGET   = 'multi-enzyme-simple.test.mgf'
MGF_PATH = TARGET
 
if not (os.path.exists(MGF_PATH) and os.path.getsize(MGF_PATH) > 1e6):
    with RemoteZip(ZIP_URL) as zf:
        src = next(n for n in zf.namelist() if TARGET in n)
        zf.extract(src, '.')
    if src != MGF_PATH and os.path.exists(src):
        shutil.move(src, MGF_PATH)
        top = src.split('/')[0]
        if os.path.isdir(top): shutil.rmtree(top, ignore_errors=True)
 
print(f'{MGF_PATH}  ({os.path.getsize(MGF_PATH)/1e6:.1f} MB)')

multi-enzyme-simple.test.mgf  (300.9 MB)


In [3]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — EDA: parse MGF + plots
# ═══════════════════════════════════════════════════════════════════
def parse_mgf(path):
    records, spec, peaks, in_s = [], {}, [], False
    with open(path, 'r', errors='replace') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            if line.upper() == 'BEGIN IONS':
                spec, peaks, in_s = {}, [], True
            elif line.upper() == 'END IONS':
                if in_s:
                    records.append({'pepmass': spec.get('_pm', 0.0),
                                    'charge':  spec.get('_ch', 1),
                                    'n_peaks': len(peaks)})
                in_s = False
            elif in_s:
                if '=' in line:
                    k, _, v = line.partition('='); k = k.strip().upper()
                    if k == 'PEPMASS': spec['_pm'] = float(v.strip().split()[0])
                    elif k == 'CHARGE': spec['_ch'] = int(_re.sub(r'[^\d]', '', v.strip()) or '1')
                else:
                    p = line.split()
                    if p:
                        try: peaks.append(float(p[0]))
                        except ValueError: pass
    return pd.DataFrame(records)
 
eda = parse_mgf(MGF_PATH)
print(f'Spectra : {len(eda):,}')
print(f'Charge  : +{eda.charge.min()} to +{eda.charge.max()} | '
      f'+2: {int((eda.charge==2).sum()):,}  +3: {int((eda.charge==3).sum()):,}')
print(f'm/z     : {eda.pepmass.min():.1f} – {eda.pepmass.max():.1f}')
print(f'Peaks   : {eda.n_peaks.mean():.0f} avg  (min {eda.n_peaks.min()}, max {eda.n_peaks.max()})')
 
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('EDA — multi-enzyme-simple.test.mgf', fontweight='bold')
vc = eda.charge.value_counts().sort_index()
axes[0].bar(vc.index.astype(str), vc.values, color='steelblue', edgecolor='white')
axes[0].set(title='Charge distribution', xlabel='Charge', ylabel='Count')
for bar, v in zip(axes[0].patches, vc.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(vc.values)*0.01,
                 f'{v:,}', ha='center', fontsize=7)
axes[1].hist(eda.pepmass.clip(upper=3000), bins=60, color='darkorange', edgecolor='white')
axes[1].set(title='Precursor m/z (clip @3000)', xlabel='m/z', ylabel='Count')
axes[2].hist(eda.n_peaks.clip(upper=500), bins=60, color='seagreen', edgecolor='white')
axes[2].set(title='Peaks/spectrum (clip @500)', xlabel='Peaks', ylabel='Count')
plt.tight_layout()
plt.savefig('results/eda_plots.png', dpi=150, bbox_inches='tight'); plt.show()
print('Saved: results/eda_plots.png')

Spectra : 106,933
Charge  : +1 to +8 | +2: 40,614  +3: 39,146
m/z     : 301.2 – 1604.3
Peaks   : 123 avg  (min 6, max 950)
Saved: results/eda_plots.png


In [4]:
 
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — Loading Casanovo model via Python API
# Uses the exact same checkpoint-finding function the CLI uses.
# ModelRunner handles all Lightning version compatibility internally.
# ═══════════════════════════════════════════════════════════════════
import appdirs
from casanovo.casanovo import _get_model_weights
from casanovo.denovo.model_runner import ModelRunner
from casanovo.config import Config
 
cache_dir = Path(appdirs.user_cache_dir("casanovo", False, opinion=False))
print(f'Cache dir : {cache_dir}')
ckpt_path  = _get_model_weights(cache_dir)   # finds cached or downloads
print(f'Checkpoint: {ckpt_path}  ({os.path.getsize(str(ckpt_path))/1e6:.0f} MB)')
 
config = Config()
runner = ModelRunner(config=config, model_filename=str(ckpt_path))
runner.initialize_tokenizer()
runner.initialize_model(train=False)
model  = runner.model.to(DEVICE).eval()
 
n_params = sum(p.numel() for p in model.parameters())
print(f'\nModel class : {type(model).__name__}')
print(f'Parameters  : {n_params/1e6:.1f}M')
print(f'dim_model   : {model.encoder.latent_spectrum.shape[-1]}')
print(f'Enc layers  : {model.encoder.transformer_encoder.num_layers}')
try:
    print(f'Dec layers  : {model.decoder.transformer_decoder.num_layers}')
except AttributeError:
    pass
print(f'n_beams     : {model.n_beams} | max_peptide_len: {model.max_peptide_len}')

Checkpoint directory not set in ModelRunner, no checkpoint files will be saved.
Configured residue(s) not in model alphabet: C[Carbamidomethyl], [Carbamyl]-, Q[Deamidated], N[Deamidated], [Ammonia-loss]-, [Acetyl]-, M[Oxidation], [+25.980265]-


Cache dir : /home/zeus/.cache/casanovo
Checkpoint: /home/zeus/.cache/casanovo/casanovo_v5_0_0_v5_0_0.ckpt  (575 MB)

Model class : Spec2Pep
Parameters  : 47.9M
dim_model   : 512
Enc layers  : 9
Dec layers  : 9
n_beams     : 1 | max_peptide_len: 100


In [5]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — Builds small MGF subset + DataLoader (bs=1)
# KEY FIX: max_charge=model.decoder.charge_encoder.num_embeddings
# ensures only charges the model supports enter the DataLoader.
# Lance is rebuilt when max_charge changes (marker file tracks this).
# ═══════════════════════════════════════════════════════════════════
from casanovo.denovo.dataloaders import DeNovoDataModule
import shutil as _shutil
from tqdm import tqdm
import threading

N_SUBSET  = 100
N_WARMUP  = 5
N_PROFILE = 7
N_TIMING  = 50

SUBSET_MGF = 'subset_profile.mgf'
LANCE_DIR  = os.path.join(os.getcwd(), 'lance_cache')
os.makedirs(LANCE_DIR, exist_ok=True)

# ── Get model's charge capacity ───────────────────────────────────
# charge_encoder = Embedding(max_charge, d_model) in transformers.py
# Valid input indices: 0 to max_charge-1 (i.e. charges 1 to max_charge)
MODEL_MAX_CHARGE = model.decoder.charge_encoder.num_embeddings
print(f'Model max_charge : {MODEL_MAX_CHARGE}  (supports charges 1–{MODEL_MAX_CHARGE})')

# ── Build subset MGF once ─────────────────────────────────────────
def write_subset_mgf(src, dest, n):
    count, buf, in_s = 0, [], False
    with open(src, 'r', errors='replace') as fin, open(dest, 'w') as fout:
        for line in fin:
            if count >= n: break
            if line.strip().upper() == 'BEGIN IONS':
                in_s = True; buf = [line]
            elif line.strip().upper() == 'END IONS':
                buf.append(line); fout.writelines(buf)
                count += 1; in_s = False; buf = []
            elif in_s:
                buf.append(line)
    return count

if not os.path.exists(SUBSET_MGF):
    wrote = write_subset_mgf(MGF_PATH, SUBSET_MGF, N_SUBSET)
    print(f'Created subset MGF: {SUBSET_MGF}  ({wrote} spectra)')
else:
    print(f'Reusing existing subset MGF: {SUBSET_MGF}')

# ── Rebuild Lance if max_charge changed since last build ──────────
# The Lance records which spectra passed charge filtering at build time.
# If it was built with a different max_charge, spectra with out-of-range
# charges would be included, causing CUDA assertion in charge_encoder.
_mc_marker  = os.path.join(LANCE_DIR, '.max_charge')
_lance_test = os.path.join(LANCE_DIR, 'test.lance')
_prev_mc    = open(_mc_marker).read().strip() if os.path.exists(_mc_marker) else 'none'

if _prev_mc != str(MODEL_MAX_CHARGE):
    if os.path.exists(_lance_test):
        _shutil.rmtree(_lance_test)
        print(f'Deleted stale Lance (was max_charge={_prev_mc})')
    with open(_mc_marker, 'w') as _f:
        _f.write(str(MODEL_MAX_CHARGE))
    print(f'Will build Lance with max_charge={MODEL_MAX_CHARGE}…')
else:
    print(f'Reusing existing Lance (max_charge={MODEL_MAX_CHARGE} verified)')

dm = DeNovoDataModule(
    lance_dir=LANCE_DIR,
    test_paths=[SUBSET_MGF],
    eval_batch_size=1,
    tokenizer=runner.model.tokenizer,
    max_charge=MODEL_MAX_CHARGE,   # ← critical: filters out charge > MODEL_MAX_CHARGE
    n_workers=0,                   # ← no forked workers (deadlock prevention)
)
dm.setup(stage='test', annotated=False)
print('DataModule ready.')

# ── Verify first batch is valid ───────────────────────────────────
_it = iter(dm.predict_dataloader())
_b  = next(_it); del _it
_mzs, _ints, _precs, _ = model._process_batch(_b)
_charge = _precs[0, 1].item()
assert _charge <= MODEL_MAX_CHARGE, \
    f'Charge {_charge} > model max_charge {MODEL_MAX_CHARGE} — Lance rebuild failed'
print(f'Batch: mzs={_mzs.shape}  precs={_precs.shape}')
print(f'Precursor: [mass={_precs[0,0]:.1f}, charge={_charge:.0f}, mz={_precs[0,2]:.1f}]  charge ✓')

Model max_charge : 4  (supports charges 1–4)
Created subset MGF: subset_profile.mgf  (100 spectra)
Will build Lance with max_charge=4…


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

DataModule ready.
Batch: mzs=torch.Size([1, 42])  precs=torch.Size([1, 3])
Precursor: [mass=3370.5, charge=3, mz=1124.5]  charge ✓


In [6]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6 — Real Spectra Stage Timing (N_TIMING spectra, bs=1)
# ═══════════════════════════════════════════════════════════════════
timings = {k: [] for k in ('fetch_ms','h2d_ms','enc_ms','dec_ms','full_decode_ms','write_ms','total_ms')}
_enc_buf = []

def _enc_pre(m, inp):
    if DEVICE == 'cuda': torch.cuda.synchronize()
    m._t0 = time.perf_counter()

def _enc_post(m, inp, out):
    if DEVICE == 'cuda': torch.cuda.synchronize()
    _enc_buf.append((time.perf_counter() - m._t0) * 1000)

_hp = model.encoder.register_forward_pre_hook(_enc_pre)
_hq = model.encoder.register_forward_hook(_enc_post)

_gpu_samples = []
_stop_gpu    = threading.Event()
def _gpu_monitor():
    while not _stop_gpu.is_set():
        r = subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used',
                            '--format=csv,noheader,nounits'], capture_output=True, text=True)
        if r.returncode == 0:
            try:
                p = r.stdout.strip().split(', ')
                _gpu_samples.append((int(p[0]), float(p[1])/1024))
            except Exception: pass
        time.sleep(0.5)
threading.Thread(target=_gpu_monitor, daemon=True).start()

loader = dm.predict_dataloader()
_iter  = iter(loader)

print(f'Warmup ({N_WARMUP} spectra)…')
with torch.no_grad():
    _w = 0
    while _w < N_WARMUP:
        try:
            _b = next(_iter)
            _mz, _it, _pr, _ = model._process_batch(_b)
            if _pr[0, 1].item() > MODEL_MAX_CHARGE:
                continue   # skip invalid charge (should not happen after Cell 5 fix)
            model.beam_search_decode(_mz.to(DEVICE), _it.to(DEVICE), _pr.to(DEVICE))
            _w += 1
        except StopIteration:
            break
if DEVICE == 'cuda': torch.cuda.synchronize()
_iter = iter(loader)   # reset for timing

def _sync():
    if DEVICE == 'cuda': torch.cuda.synchronize()

n_timed = min(N_TIMING, N_SUBSET)
pbar = tqdm(range(n_timed), desc='Timing real spectra', unit='spec',
            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining} {rate_fmt}]')
for _ in pbar:
    _enc_buf.clear()
    try:
        _sync(); t0 = time.perf_counter()
        batch = next(_iter)
        t_fetch = (time.perf_counter() - t0) * 1000

        _sync(); t0 = time.perf_counter()
        mzs_r, ints_r, precs_r, _ = model._process_batch(batch)
        if precs_r[0, 1].item() > MODEL_MAX_CHARGE:
            continue   # safety guard
        mzs_r   = mzs_r.to(DEVICE)
        ints_r  = ints_r.to(DEVICE)
        precs_r = precs_r.to(DEVICE)
        _sync(); t_h2d = (time.perf_counter() - t0) * 1000

        _sync(); t0 = time.perf_counter()
        with torch.no_grad():
            preds = model.beam_search_decode(mzs_r, ints_r, precs_r)
        _sync(); t_full = (time.perf_counter() - t0) * 1000

        t_enc = _enc_buf[0] if _enc_buf else 0.0
        t_dec = max(0.0, t_full - t_enc)

        t0 = time.perf_counter()
        _out = [{'peptide': pep, 'score': float(sc)} for sp in preds for sc, _, pep in sp]
        t_write = (time.perf_counter() - t0) * 1000

        t_total = t_fetch + t_h2d + t_full + t_write
        for k, v in zip(timings.keys(),
                        (t_fetch, t_h2d, t_enc, t_dec, t_full, t_write, t_total)):
            timings[k].append(v)

        pbar.set_postfix({'total_ms': f'{t_total:.0f}', 'enc_ms': f'{t_enc:.0f}'})
    except StopIteration:
        break

_stop_gpu.set()
_hp.remove(); _hq.remove()

n = len(timings['total_ms'])
df_real = pd.DataFrame([
    {'Stage': 'DataLoader fetch',  'mean_ms': np.mean(timings['fetch_ms']),
     'p50_ms': np.percentile(timings['fetch_ms'],  50), 'p95_ms': np.percentile(timings['fetch_ms'],  95)},
    {'Stage': 'H2D transfer',      'mean_ms': np.mean(timings['h2d_ms']),
     'p50_ms': np.percentile(timings['h2d_ms'],    50), 'p95_ms': np.percentile(timings['h2d_ms'],    95)},
    {'Stage': 'SpectrumEncoder',   'mean_ms': np.mean(timings['enc_ms']),
     'p50_ms': np.percentile(timings['enc_ms'],    50), 'p95_ms': np.percentile(timings['enc_ms'],    95)},
    {'Stage': 'AR decode (total)', 'mean_ms': np.mean(timings['dec_ms']),
     'p50_ms': np.percentile(timings['dec_ms'],    50), 'p95_ms': np.percentile(timings['dec_ms'],    95)},
    {'Stage': 'Output write',      'mean_ms': np.mean(timings['write_ms']),
     'p50_ms': np.percentile(timings['write_ms'],  50), 'p95_ms': np.percentile(timings['write_ms'],  95)},
    {'Stage': 'TOTAL per spectrum','mean_ms': np.mean(timings['total_ms']),
     'p50_ms': np.percentile(timings['total_ms'],  50), 'p95_ms': np.percentile(timings['total_ms'],  95)},
]).round(2)

throughput = 1000 / np.mean(timings['total_ms'])
gpu_util   = np.mean([s[0] for s in _gpu_samples]) if _gpu_samples else 0
gpu_vram   = np.max([s[1]  for s in _gpu_samples]) if _gpu_samples else 0

print(f'\n── Stage breakdown ({n} real spectra, bs=1) ──')
print(df_real.to_string(index=False))
print(f'\nThroughput : {throughput:.1f} spec/s  |  GPU {gpu_util:.0f}%  |  VRAM {gpu_vram:.2f} GB')

# Pre-compute to avoid f-string backslash restriction (Python 3.10)
mean_t     = np.mean(timings['total_ms'])
target_msg = "MEETS ✓" if mean_t <= 35 else f"FAILS — {mean_t:.0f} ms ({mean_t/35:.1f}x over)"
print(f'35 ms target: {target_msg}')
df_real.to_csv('results/real_spectra_timing.csv', index=False)

Warmup (5 spectra)…


Timing real spectra: 100%|██████████| 50/50 [00:18<00:00  2.66spec/s]


── Stage breakdown (50 real spectra, bs=1) ──
             Stage  mean_ms  p50_ms  p95_ms
  DataLoader fetch     1.81    1.72    2.17
      H2D transfer     0.25    0.24    0.30
   SpectrumEncoder     8.22    8.13    9.06
 AR decode (total)   363.90  337.14  599.07
      Output write     0.01    0.01    0.01
TOTAL per spectrum   374.19  347.06  608.76

Throughput : 2.7 spec/s  |  GPU 12%  |  VRAM 0.44 GB
35 ms target: FAILS — 374 ms (10.7x over)


In [7]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7 — torch.profiler on Real Spectra (7 batches, bs=1)
# FIX: validate all pre-fetched batches for charge <= MODEL_MAX_CHARGE
#      before entering the profiler. This prevents the CUDA assertion
#      from charge_encoder(out_of_range_index) firing inside the profiler.
# profile_memory=False: kept off to avoid allocation hook interference.
# key_averages() called inside on_trace_ready (before buffer reset).
# ═══════════════════════════════════════════════════════════════════
ACTS     = [ProfilerActivity.CPU, ProfilerActivity.CUDA] if DEVICE == 'cuda' else [ProfilerActivity.CPU]
SORT_KEY = 'cpu_time_total'

# ── Pre-fetch and validate batches ───────────────────────────────
print(f'Pre-fetching real batches (max_charge={MODEL_MAX_CHARGE})…')
_real_batches = []
_skipped      = 0
for _b in dm.predict_dataloader():
    _mz, _it, _pr, _ = model._process_batch(_b)
    _charge = _pr[0, 1].item()
    if _charge > MODEL_MAX_CHARGE:
        _skipped += 1
        continue   # skip — would cause charge_encoder out-of-bounds
    _real_batches.append((_mz.to(DEVICE), _it.to(DEVICE), _pr.to(DEVICE)))
    if len(_real_batches) >= N_PROFILE:
        break

if _skipped:
    print(f'Skipped {_skipped} spectra with charge > {MODEL_MAX_CHARGE}')

if len(_real_batches) < 3:
    raise RuntimeError(
        f'Only {len(_real_batches)} valid batches found (need ≥ 3). '
        f'Increase N_SUBSET in Cell 5.')

print(f'Fetched {len(_real_batches)} valid batches — '
      f'charges: {[int(b[2][0,1].item()) for b in _real_batches]}')

# Warmup on real data
with torch.no_grad():
    for _mz, _it, _pr in _real_batches[:2]:
        model.beam_search_decode(_mz, _it, _pr)
if DEVICE == 'cuda': torch.cuda.synchronize()

# ── Profiler A: SpectrumEncoder ───────────────────────────────────
print('\n── torch.profiler: SpectrumEncoder (real spectra, bs=1) ──')
_enc_store = {}
def _enc_ready(p):
    p.export_chrome_trace('results/trace_real_encoder.json')
    _enc_store['tbl'] = p.key_averages().table(sort_by=SORT_KEY, row_limit=10)

with profile(activities=ACTS, record_shapes=True,
             schedule=schedule(wait=0, warmup=2, active=5),
             on_trace_ready=_enc_ready) as p_enc:
    with torch.no_grad():
        for _mz, _it, _pr in _real_batches:
            with record_function('encoder_real'):
                model.encoder(_mz, _it)
            if DEVICE == 'cuda': torch.cuda.synchronize()
            p_enc.step()

print(_enc_store.get('tbl', '(no data — see trace file)'))
with open('results/profiler_real_encoder.txt', 'w') as fh:
    fh.write('torch.profiler — SpectrumEncoder real spectra (bs=1)\n' + '='*60 + '\n')
    fh.write(str(_enc_store.get('tbl', 'no data')))
print('Trace → results/trace_real_encoder.json  (ui.perfetto.dev)')

if DEVICE == 'cuda':
    torch.cuda.synchronize()
    torch.cuda.empty_cache()

# ── Profiler B: full beam_search_decode ───────────────────────────
print('\n── torch.profiler: beam_search_decode (real spectra, bs=1) ──')
_dec_store = {}
def _dec_ready(p):
    p.export_chrome_trace('results/trace_real_decode.json')
    _dec_store['tbl'] = p.key_averages().table(sort_by=SORT_KEY, row_limit=12)

with profile(activities=ACTS, record_shapes=True,
             schedule=schedule(wait=0, warmup=2, active=5),
             on_trace_ready=_dec_ready) as p_dec:
    with torch.no_grad():
        for _mz, _it, _pr in _real_batches:
            with record_function('beam_search_decode_real'):
                model.beam_search_decode(_mz, _it, _pr)
            if DEVICE == 'cuda': torch.cuda.synchronize()
            p_dec.step()

print(_dec_store.get('tbl', '(no data — see trace file)'))
with open('results/profiler_real_decode.txt', 'w') as fh:
    fh.write('torch.profiler — beam_search_decode real spectra (bs=1)\n' + '='*60 + '\n')
    fh.write(str(_dec_store.get('tbl', 'no data')))
print('Trace → results/trace_real_decode.json  (ui.perfetto.dev)')

if DEVICE == 'cuda' and _dec_store.get('tbl'):
    try:
        _ops = sorted(
            [(e.key, e.cuda_time_total/1000, e.count)
             for e in p_dec.key_averages() if e.cuda_time_total > 0],
            key=lambda x: -x[1])
        if _ops:
            _tot = sum(t for _, t, _ in _ops) or 1
            print('\n── CUDA kernel breakdown ──')
            print(f'{"Kernel":<50} {"ms":>7} {"n":>5} {"%":>6}')
            print('-' * 70)
            for kn, kms, cnt in _ops[:10]:
                print(f'{kn[:50]:<50} {kms:>7.2f} {cnt:>5} {kms/_tot*100:>5.1f}%')
        else:
            print('CUDA kernels: see trace file at ui.perfetto.dev')
    except Exception as _ke:
        print(f'CUDA breakdown note: {_ke}')

Pre-fetching real batches (max_charge=4)…
Fetched 7 valid batches — charges: [3, 4, 4, 3, 4, 2, 4]

── torch.profiler: SpectrumEncoder (real spectra, bs=1) ──
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.65%     582.362us       100.00%      89.414ms      17.883ms       0.000us         0.00%       5.856ms       1.171ms             5  
                                           encod

In [8]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8 — Synthetic Micro-benchmark (isolated, no DataLoader noise)
# Clean baseline for kernel-level comparison with NAR (PR #548).
# Uses fixed 123-peak synthetic batch matching dataset statistics.
# ═══════════════════════════════════════════════════════════════════
def make_synth_batch(device=DEVICE):
    n_real = 123
    mzs_s  = torch.zeros(1, N_PEAKS, device=device)
    ints_s = torch.zeros(1, N_PEAKS, device=device)
    mzs_s[0,  :n_real] = torch.rand(n_real, device=device) * 1303 + 301
    ints_s[0, :n_real] = torch.rand(n_real, device=device)
    ints_s[0, :n_real] /= ints_s[0, :n_real].norm()
    charge = 2.0; pmz = 600.0
    precs  = torch.tensor([[(pmz-1.007276)*charge, charge, pmz]], device=device)
    return mzs_s, ints_s, precs
 
smzs, sints, sprecs = make_synth_batch()
sempty = torch.zeros(1, 0, dtype=torch.int64, device=DEVICE)
 
with torch.no_grad():
    for _ in range(5): model.beam_search_decode(smzs, sints, sprecs)
if DEVICE == 'cuda': torch.cuda.synchronize()
with torch.no_grad(): smems, smasks = model.encoder(smzs, sints)
if DEVICE == 'cuda': torch.cuda.synchronize()
 
# Micro-benchmark: encoder
with profile(activities=ACTS, record_shapes=True,
             schedule=schedule(wait=0, warmup=2, active=5),
             on_trace_ready=lambda p: p.export_chrome_trace(
                 'results/trace_synth_encoder.json')) as p_se:
    with torch.no_grad():
        for _ in range(7):
            with record_function('encoder_synth'): model.encoder(smzs, sints)
            if DEVICE == 'cuda': torch.cuda.synchronize()
            p_se.step()
 
# Micro-benchmark: single decoder step
with profile(activities=ACTS, record_shapes=True,
             schedule=schedule(wait=0, warmup=2, active=5),
             on_trace_ready=lambda p: p.export_chrome_trace(
                 'results/trace_synth_decoder_step.json')) as p_sd:
    with torch.no_grad():
        for _ in range(7):
            with record_function('decoder_step_synth'):
                model.decoder(tokens=sempty, memory=smems,
                               memory_key_padding_mask=smasks, precursors=sprecs)
            if DEVICE == 'cuda': torch.cuda.synchronize()
            p_sd.step()
 
with open('results/profiler_synth_encoder.txt', 'w') as fh:
    fh.write(p_se.key_averages().table(sort_by=SORT_KEY, row_limit=10))
with open('results/profiler_synth_decoder_step.txt', 'w') as fh:
    fh.write(p_sd.key_averages().table(sort_by=SORT_KEY, row_limit=10))
 
# Sync-accurate timings (20 reps, first 5 discarded)
def _sync():
    if DEVICE == 'cuda': torch.cuda.synchronize()
 
_et, _dt, _ft = [], [], []
with torch.no_grad():
    for _ in range(20):
        _sync(); t0 = time.perf_counter(); model.encoder(smzs, sints)
        _sync(); _et.append((time.perf_counter()-t0)*1000)
    for _ in range(20):
        _sync(); t0 = time.perf_counter()
        model.decoder(tokens=sempty, memory=smems, memory_key_padding_mask=smasks, precursors=sprecs)
        _sync(); _dt.append((time.perf_counter()-t0)*1000)
    for _ in range(20):
        _sync(); t0 = time.perf_counter(); model.beam_search_decode(smzs, sints, sprecs)
        _sync(); _ft.append((time.perf_counter()-t0)*1000)
 
enc_s = float(np.mean(_et[5:])); dec_s = float(np.mean(_dt[5:])); full_s = float(np.mean(_ft[5:]))
nar_s = enc_s + dec_s
meas_T = round((full_s - enc_s) / dec_s) if dec_s > 0 else 12
 
print(f'\n── Synthetic micro timings (isolated, 20 reps) ──')
print(f'SpectrumEncoder     : {enc_s:.2f} ms  ({enc_s/full_s*100:.0f}%)')
print(f'Decoder × 1 step    : {dec_s:.2f} ms/step')
print(f'beam_search_decode  : {full_s:.2f} ms  (~{meas_T} steps)')
print(f'NAR projected †     : {nar_s:.2f} ms  →  {1000/nar_s:.1f} spec/s')
print('† Projection assumes 1 parallel decoder pass (PR #548 needed for validation)')
print('Traces → results/trace_synth_encoder.json  |  trace_synth_decoder_step.json')


── Synthetic micro timings (isolated, 20 reps) ──
SpectrumEncoder     : 8.76 ms  (3%)
Decoder × 1 step    : 14.36 ms/step
beam_search_decode  : 263.94 ms  (~18 steps)
NAR projected †     : 23.12 ms  →  43.3 spec/s
† Projection assumes 1 parallel decoder pass (PR #548 needed for validation)
Traces → results/trace_synth_encoder.json  |  trace_synth_decoder_step.json


In [9]:
# ═══════════════════════════════════════════════════════════════════
# CELL 9 — Plots + summary.txt
# ═══════════════════════════════════════════════════════════════════
import datetime
 
mean_total = np.mean(timings['total_ms'])
p50_total  = np.percentile(timings['total_ms'], 50)
p95_total  = np.percentile(timings['total_ms'], 95)
 
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'Casanovo Phase 1 Profiling — bs=1 | {GPU_NAME}', fontweight='bold')
 
# Panel 1: stage breakdown (real spectra)
s_labels = ['Fetch', 'H2D', 'Encoder', 'AR Decode', 'Write']
s_means  = [np.mean(timings[k]) for k in ('fetch_ms','h2d_ms','enc_ms','dec_ms','write_ms')]
bars = axes[0].bar(s_labels, s_means,
                   color=['#888','#E67E22','#378ADD','#D85A30','#888'],
                   edgecolor='none', width=0.55)
for bar, v in zip(bars, s_means):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{v:.1f}', ha='center', fontsize=9)
axes[0].set_title(f'Stage breakdown\n({n} real spectra, bs=1)')
axes[0].set_ylabel('ms'); axes[0].spines[['top','right']].set_visible(False)
 
# Panel 2: latency distribution
axes[1].hist(timings['total_ms'], bins=15, color='#378ADD', edgecolor='white', alpha=0.85)
axes[1].axvline(mean_total, color='#D85A30', lw=2, ls='--', label=f'mean {mean_total:.0f} ms')
axes[1].axvline(p50_total,  color='#1D9E75', lw=2, ls='-.',  label=f'p50  {p50_total:.0f} ms')
axes[1].axvline(p95_total,  color='#9B59B6', lw=1.5, ls=':', label=f'p95  {p95_total:.0f} ms')
axes[1].axvline(35, color='black', lw=1, ls='--', alpha=0.5, label='35 ms target')
axes[1].legend(fontsize=8, frameon=False)
axes[1].set_title('Latency distribution (bs=1)')
axes[1].set_xlabel('ms/spectrum'); axes[1].spines[['top','right']].set_visible(False)
 
# Panel 3: AR vs NAR vs targets (synthetic timings)
ax3 = axes[2]
bar3 = ax3.bar(['AR\n(measured)', 'NAR\n(projected†)'], [full_s, nar_s],
               color=['#D85A30','#1D9E75'], edgecolor='none', width=0.35)
for bar, v in zip(bar3, [full_s, nar_s]):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
             f'{v:.0f} ms\n{1000/v:.1f} s/s', ha='center', fontsize=9)
ax3.axhline(35,  color='#9B59B6', lw=2,   ls='--', label='35 ms target')
ax3.axhline(50,  color='#E67E22', lw=1.5, ls='-.', label='50 ms ok')
ax3.axhline(100, color='#888',    lw=1,   ls=':',  label='10 Hz')
ax3.legend(fontsize=8, frameon=False)
ax3.set_title('AR vs NAR (synthetic micro-timing)')
ax3.set_ylabel('ms'); ax3.spines[['top','right']].set_visible(False)
fig.text(0.5, -0.03, '† NAR: 1 encoder + 1 parallel decoder pass — requires PR #548',
         ha='center', fontsize=8, color='gray', style='italic')
plt.tight_layout()
plt.savefig('results/profiling_summary.png', dpi=150, bbox_inches='tight'); plt.show()
print('Saved: results/profiling_summary.png')
 
# ── summary.txt ──────────────────────────────────────────────────
summary = f"""CASANOVO PHASE 1 PROFILING — bs=1 REAL-TIME ANALYSIS
Generated : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}
Hardware  : {GPU_NAME} | {TOTAL_VRAM:.1f} GB VRAM
Software  : Casanovo {type(model).__name__} | PyTorch {torch.__version__} | CUDA {torch.version.cuda}
Dataset   : {SUBSET_MGF} ({N_SUBSET} spectra subset from {MGF_PATH})
 
CONTEXT
  Instrument rate : ~20 Hz → 50 ms per spectrum budget
  Only bs=1 is meaningful for real-time (per mentor discussion).
  Target: <35 ms (good) | <50 ms (ok) — from Devin's message.
 
REAL SPECTRA TIMING ({n} spectra, bs=1)
{df_real.to_string(index=False)}
  Throughput : {1000/mean_total:.1f} spec/s
  GPU util   : {gpu_util:.0f}%  |  VRAM: {gpu_vram:.2f} / {TOTAL_VRAM:.1f} GB
  35 ms target: {"MEETS ✓" if mean_total <= 35 else f"FAILS — {mean_total:.0f} ms ({mean_total/35:.1f}x over)"}
 
SYNTHETIC MICRO TIMINGS (20 reps, isolated)
  SpectrumEncoder         : {enc_s:.2f} ms  ({enc_s/full_s*100:.0f}%)
  PeptideDecoder ×1 step  : {dec_s:.2f} ms/step
  beam_search_decode total: {full_s:.2f} ms  (~{meas_T} AR steps)
  NAR projected (1 pass) †: {nar_s:.2f} ms → {1000/nar_s:.1f} spec/s
  † Requires PR #548 for validation.
 
BOTTLENECKS (Phase 1 → Phase 2 priorities)
  1. AR sequential decoder: {(full_s-enc_s)/full_s*100:.0f}% of total | ~{meas_T} passes/spectrum
     Fix: NAR (PR #548) — eliminates T-step sequential dependency
  2. Remaining gap after NAR: {max(0,nar_s-35):.0f} ms above 35 ms target
     Fix: CUDA Graphs + BF16 + FlashAttention-2
  3. Encoder: {enc_s/full_s*100:.0f}% — fixed cost; gains from BF16 + torch.compile
 
ARTIFACTS
  results/trace_real_encoder.json        ← SpectrumEncoder (real spectra)
  results/trace_real_decode.json         ← full decode (real spectra)
  results/trace_synth_encoder.json       ← encoder micro-kernel (synthetic)
  results/trace_synth_decoder_step.json  ← decoder single step (synthetic)
  Open .json at ui.perfetto.dev
  results/profiler_real_encoder.txt  profiler_real_decode.txt  profiler_real_memory.txt
  results/profiler_synth_encoder.txt  profiler_synth_decoder_step.txt
  results/real_spectra_timing.csv  profiling_summary.png  eda_plots.png
"""
print(summary)
with open('results/summary.txt', 'w') as fh: fh.write(summary)
pd.DataFrame({'metric':['mean_ms','p50_ms','p95_ms','throughput','gpu_pct','vram_gb'],
              'value':[round(mean_total,1),round(p50_total,1),round(p95_total,1),
                       round(1000/mean_total,1),round(gpu_util,0),round(gpu_vram,2)]}
             ).to_csv('results/benchmark_table.csv', index=False)
 
print('\n── results/ ──')
for f in sorted(os.listdir('results')):
    fp = os.path.join('results', f)
    print(f'  {f:<50} {os.path.getsize(fp)/1024:.1f} KB')
print('\nPhase 1 complete. Primary deliverable: results/summary.txt')

Saved: results/profiling_summary.png
CASANOVO PHASE 1 PROFILING — bs=1 REAL-TIME ANALYSIS
Generated : 2026-05-24 12:08
Hardware  : NVIDIA L4 | 23.6 GB VRAM
Software  : Casanovo Spec2Pep | PyTorch 2.7.1+cu128 | CUDA 12.8
Dataset   : subset_profile.mgf (100 spectra subset from multi-enzyme-simple.test.mgf)
 
CONTEXT
  Instrument rate : ~20 Hz → 50 ms per spectrum budget
  Only bs=1 is meaningful for real-time (per mentor discussion).
  Target: <35 ms (good) | <50 ms (ok) — from Devin's message.
 
REAL SPECTRA TIMING (50 spectra, bs=1)
             Stage  mean_ms  p50_ms  p95_ms
  DataLoader fetch     1.81    1.72    2.17
      H2D transfer     0.25    0.24    0.30
   SpectrumEncoder     8.22    8.13    9.06
 AR decode (total)   363.90  337.14  599.07
      Output write     0.01    0.01    0.01
TOTAL per spectrum   374.19  347.06  608.76
  Throughput : 2.7 spec/s
  GPU util   : 12%  |  VRAM: 0.44 / 23.6 GB
  35 ms target: FAILS — 374 ms (10.7x over)
 
SYNTHETIC MICRO TIMINGS (20 reps, iso